<a href="https://colab.research.google.com/github/Fiona-ali/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This draft is for the Week 3 **Refresh / Content Opportunity Scoring** lane carried forward from ML-03. It is not ready to submit until the gated Hugging Face queries run successfully and their outputs are saved. It uses March 2026 as a mid-panel feature window and keeps April only for the future outcome label. Never put the Hugging Face token in this public notebook.


## 1. Unit of analysis + time window

**One row = one pseudonymized client–content pair at the March 31, 2026 decision cutoff.** Features come only from daily GSC data for March 1–31, 2026. The outcome proxy uses April 1–30, 2026. Keep pairs with at least 100 March impressions and GSC availability marked true in both months.

**Tables:** `fact_content_daily_performance` for March features and April outcome; no query-level table or June sample.

**Lane decision:** rank pages for an editor to review for a possible refresh. The label is an observed next-month impressions decline greater than 20% versus March, not evidence that a refresh would recover traffic. The queue supports human prioritization; it does not edit or publish content.


In [3]:
%pip -q install duckdb

import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Read the token from Colab Secrets or an environment variable.
# Never paste or print it in this notebook or the public repository.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Hugging Face READ token as a Colab Secret named HF_TOKEN, "
            "then rerun this cell. Do not put the token in a code cell."
        ) from exc
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add it in Colab Secrets.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
del safe_token, HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH_REL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL_REL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
MARCH_START = "2026-03-01"
MARCH_END_EXCLUSIVE = "2026-04-01"
APRIL_START = "2026-04-01"
APRIL_END_EXCLUSIVE = "2026-05-01"


## 2. Fields: feature / label / context / excluded

Five candidate features, all knowable by the end-of-March decision:

1. `impressions_march` — March GSC impressions; visibility scale.
2. `clicks_march` — March GSC clicks; realized search visits.
3. `mean_daily_position_march` — mean of non-null daily GSC average positions on impression days; lower is better.
4. `active_gsc_days_march` — distinct March days with at least one impression; activity/coverage.
5. `ctr_march_pct` — March clicks divided by March impressions × 100; the 100-impression floor reduces low-volume noise.

**Label/proxy:** `decline_april_vs_march` = 1 when April impressions are more than 20% below March; otherwise 0. The April aggregate is outcome-only, never a predictor.

**Context:** `client_hash_id` and `content_hash_id` identify/group page pairs and support a client-held-out split; `report_date` defines the windows; `gsc_data_available` is only an access/coverage filter. None is a feature.

**Excluded:** all April metrics from predictors (future information); `fact_content_daily_performance_sample` (June 2026 is the final sealed month); `fact_content_query_90d` (its fixed window can overlap outcome periods); IDs as predictors; GA4 metrics (not needed for this GSC-first lane and valid only when their separate availability flag is true); product-decision flags and any identifying fields not present in the public release.


In [4]:
# Field inventory: this is the planned contract, not warehouse evidence.
FEATURES = [
    "impressions_march",
    "clicks_march",
    "mean_daily_position_march",
    "active_gsc_days_march",
    "ctr_march_pct",
]
LABEL = "decline_april_vs_march"
CONTEXT = ["client_hash_id", "content_hash_id"]
print("Candidate features:", FEATURES)
print("Outcome-only label:", LABEL)
print("Context only:", CONTEXT)


Candidate features: ['impressions_march', 'clicks_march', 'mean_daily_position_march', 'active_gsc_days_march', 'ctr_march_pct']
Outcome-only label: decline_april_vs_march
Context only: ['client_hash_id', 'content_hash_id']


## 3. Verify it with three queries (grain, counts, availability)

The first three query outputs are the required contract checks. The feature query then builds one row per eligible client–content pair. It aggregates only March and April month partitions; April stays in the label column.


In [6]:
# Query 1 — daily fact grain: one row per date × client × content.
grains = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {MARCH_REL}
        WHERE report_date >= DATE '{MARCH_START}'
          AND report_date <  DATE '{MARCH_END_EXCLUSIVE}'
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
display(grains)
assert int(grains.loc[0, "duplicate_grain_groups"]) == 0, "Unexpected duplicate daily grain."

# Query 2 — March slice row count and date span.
march_counts = con.sql(f"""
    SELECT COUNT(*) AS march_fact_rows,
           (SELECT COUNT(*) FROM (
               SELECT DISTINCT client_hash_id, content_hash_id
               FROM {MARCH_REL}
               WHERE report_date >= DATE '{MARCH_START}'
                 AND report_date < DATE '{MARCH_END_EXCLUSIVE}'
           )) AS client_content_pairs,
           MIN(report_date) AS first_report_date,
           MAX(report_date) AS last_report_date
    FROM {MARCH_REL}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date <  DATE '{MARCH_END_EXCLUSIVE}'
""").df()
display(march_counts)
assert pd.to_datetime(march_counts.loc[0, "first_report_date"]).strftime("%Y-%m-%d") == MARCH_START
assert pd.to_datetime(march_counts.loc[0, "last_report_date"]).strftime("%Y-%m-%d") == "2026-03-31"

# Query 3 — coverage. IS TRUE excludes false and NULL flags.
march_availability = con.sql(f"""
    SELECT COUNT(*) AS march_fact_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_true_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
           COUNT(DISTINCT client_hash_id) FILTER
               (WHERE gsc_data_available IS TRUE) AS clients_with_gsc
    FROM {MARCH_REL}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date <  DATE '{MARCH_END_EXCLUSIVE}'
""").df()
display(march_availability)
assert int(march_availability.loc[0, "gsc_available_true_rows"]) > 0

# Build the lane frame. March is the feature window; April is label-only.
feature_frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march,
               AVG(gsc_avg_position) FILTER (
                   WHERE gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
               ) AS mean_daily_position_march,
               COUNT(DISTINCT report_date) FILTER (
                   WHERE gsc_impressions > 0
               ) AS active_gsc_days_march
        FROM {MARCH_REL}
        WHERE report_date >= DATE '{MARCH_START}'
          AND report_date <  DATE '{MARCH_END_EXCLUSIVE}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS impressions_april
        FROM {APRIL_REL}
        WHERE report_date >= DATE '{APRIL_START}'
          AND report_date <  DATE '{APRIL_END_EXCLUSIVE}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT m.client_hash_id,
           m.content_hash_id,
           m.impressions_march,
           m.clicks_march,
           m.mean_daily_position_march,
           m.active_gsc_days_march,
           100.0 * m.clicks_march / NULLIF(m.impressions_march, 0) AS ctr_march_pct,
           a.impressions_april,
           CASE WHEN a.impressions_april < 0.8 * m.impressions_march
                THEN 1 ELSE 0 END AS decline_april_vs_march
    FROM march AS m
    INNER JOIN april AS a USING (client_hash_id, content_hash_id)
    WHERE m.impressions_march >= 100
""").df()

print(f"Eligible client–content pairs: {len(feature_frame):,}")
print("April decline proxy counts (0=no >20% drop, 1=>20% drop):")
print(feature_frame[LABEL].value_counts(dropna=False).sort_index())
# Show feature values only; never print pseudonymous identifiers.
display(feature_frame[FEATURES].head(5))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


,march_fact_rows,client_content_pairs,first_report_date,last_report_date
0,9841378,331437,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_fact_rows,gsc_available_true_rows,gsc_not_true_rows,clients_with_gsc
0,9841378,3611061,6230317,47


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible client–content pairs: 100,893
April decline proxy counts (0=no >20% drop, 1=>20% drop):
decline_april_vs_march
0    48949
1    51944
Name: count, dtype: int64


,impressions_march,clicks_march,mean_daily_position_march,active_gsc_days_march,ctr_march_pct
0,1423.0,1.0,16.669625,31,0.070274
1,208.0,0.0,31.880538,27,0.000000
2,607.0,0.0,30.986972,31,0.000000
3,267.0,0.0,24.274523,27,0.000000
4,1489.0,0.0,47.826506,31,0.000000


## 4. Data limits + deliberate leakage demonstration

**Limitations:** this is one March-to-April transition, not evidence across seasons. Requiring true GSC coverage and at least 100 March impressions selects measurable pages; excluded pages are not assumed to have zero impressions. A greater-than-20% fall is an observed proxy, not a causal effect of refreshing content. The fixed 90-day query table is deliberately unused because its dates can overlap outcomes. The June `_sample` partition is reserved as a sealed future window.

The code below uses a client-held-out split and Precision@50, matching the review-queue decision. It then adds a copy of the target as a feature on purpose; the resulting near-perfect score is invalid. That leak is removed and the final feature list remains at five.


In [7]:
# Honest quick check on held-out clients, then one deliberate label leak.
assert len(feature_frame) >= 100, "Too few eligible rows for a meaningful top-50 check."
X = feature_frame[FEATURES].replace([np.inf, -np.inf], np.nan)
y = feature_frame[LABEL].astype(int)
groups = feature_frame["client_hash_id"]  # grouping only, never in X

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2

# Imputation medians are learned from training clients only.
medians = X.iloc[train_idx].median().fillna(0)
X_train = X.iloc[train_idx].fillna(medians).fillna(0)
X_test = X.iloc[test_idx].fillna(medians).fillna(0)
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def precision_at_k(y_true, scores, k=50):
    k = min(k, len(y_true))
    top = np.argsort(np.asarray(scores))[::-1][:k]
    return float(np.asarray(y_true)[top].mean())

honest_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42)
honest_model.fit(X_train, y_train)
honest_scores = honest_model.predict_proba(X_test)[:, list(honest_model.classes_).index(1)]
honest_p50 = precision_at_k(y_test, honest_scores, 50)
print(f"Honest held-out-client Precision@50 (quick check): {honest_p50:.3f}")

# Deliberately leak the target itself as an extra feature, then measure on same clients.
X_train_leaky = np.column_stack([X_train, y_train.to_numpy()])
X_test_leaky = np.column_stack([X_test, y_test.to_numpy()])
leaky_model = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_model.fit(X_train_leaky, y_train)
leaky_scores = leaky_model.predict_proba(X_test_leaky)[:, list(leaky_model.classes_).index(1)]
leaky_p50 = precision_at_k(y_test, leaky_scores, 50)
leaky_accuracy = accuracy_score(y_test, leaky_model.predict(X_test_leaky))
print(f"INTENTIONALLY INVALID label-leak Precision@50: {leaky_p50:.3f}")
print(f"INTENTIONALLY INVALID label-leak accuracy: {leaky_accuracy:.3f}")

del X_train_leaky, X_test_leaky, leaky_model, leaky_scores
assert len(FEATURES) == 5
assert "client_hash_id" not in FEATURES and "content_hash_id" not in FEATURES
print("Leak removed; final candidate feature count:", len(FEATURES))


Honest held-out-client Precision@50 (quick check): 0.660
INTENTIONALLY INVALID label-leak Precision@50: 1.000
INTENTIONALLY INVALID label-leak accuracy: 1.000
Leak removed; final candidate feature count: 5


## Self-check

**Do not mark this complete yet.** After HF access is accepted and `HF_TOKEN` is stored in Colab Secrets, run every cell top to bottom. Save the executed notebook with query outputs visible, then commit it here.

- [ ] Every contract section is filled and backed by queries
- [ ] Notebook runs top to bottom with no errors
- [ ] Three query outputs are visible; grain duplicate count is zero; dates span March 2026
- [ ] Availability query filters with `IS TRUE`
- [ ] Exactly five candidate features; April appears only as outcome/label
- [ ] Intentional leak is labeled invalid and removed from final predictors
- [ ] No client names, URLs, or private queries appear in outputs
- [ ] Executed notebook committed to `work/notebooks/w03_data_contract.ipynb`
